<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_03_target_definition/stage_03b_target_predefinition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03b_target_predefinition**


## **Configuración del Entorno**


### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

### 0.3. Definición de rutas

In [3]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [4]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday.parquet"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday_labeled.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/target_definitio_summary.json"))

In [5]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### 0.4. Carga de dataset `intraday_mnq`


In [8]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [9]:
def add_column_date(df):
    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [10]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

In [11]:
mnq_intraday = load_mnq_parquet()
mnq_intraday = add_column_date(mnq_intraday)
mnq_intraday.head()

Archivo encontrado en disco. Cargando dataset local...


,date,open,high,low,close,volume
datetime,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3


In [12]:
info_dataset(mnq_intraday)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 571
	Hora diaria de inicio 06:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York


# **1. Análisis del nivel del índice (`close`) como referencia de escala**




Antes de avanzar al análisis estadístico de los retornos definidos como target de predicción, resulta necesario analizar la variable `close`, que representa el nivel del índice MNQ en cada instante temporal del dataset intradía.

A diferencia de enfoques anteriores, en esta etapa **no se definen objetivos operativos ni umbrales económicos a priori**. Sin embargo, el análisis de `close` sigue siendo metodológicamente relevante por los siguientes motivos

## **1.1. Relación entre retornos y nivel del índice**


El target de predicción se ha definido como el **retorno simple**:

$$
r_{t,h} = \frac{close_{t+h} - close_t}{close_t}
$$

Por definición, el retorno es una magnitud **relativa**, normalizada por el nivel del precio. Esto implica que:

- el mismo movimiento absoluto en puntos del índice produce retornos distintos según el valor de `close_t`,
- la escala económica real del error del modelo depende del nivel del índice en cada instante.

Por lo tanto, comprender la distribución y evolución temporal de `close` es necesario para interpretar correctamente los retornos y sus errores asociados.

## **1.2. `close` como puente entre espacio estadístico y espacio económico**


Aunque el entrenamiento de los modelos se realiza en el espacio de los retornos, la evaluación final y la interpretación operativa se realizan en **puntos del índice**.

La relación entre ambos espacios viene dada por:

$$
\Delta P_{t,h} \approx r_{t,h} \cdot close_t
$$

En este sentido, la variable `close` cumple el rol de **factor de escala** que permite:

- convertir errores relativos en errores económicos,
- comparar modelos de forma justa en términos de impacto real,
- evitar interpretaciones engañosas basadas únicamente en métricas adimensionales.


## **1.3. Robustez temporal y cambios de régimen de precios**


El dataset abarca múltiples años y regímenes de mercado, durante los cuales el nivel del índice MNQ varía significativamente. Analizar `close` permite:

- constatar que el modelo no depende implícitamente de un nivel de precios específico,
- validar la conveniencia del uso de retornos como target invariante al nivel del índice,
- anticipar posibles efectos de amplificación del error económico en períodos de precios elevados.

Este análisis es especialmente relevante para evaluar la **capacidad de generalización temporal** del enfoque adoptado.

## **1.4. Rol de este paso dentro del pipeline**

El análisis de la variable `close` cumple una función **conceptual y metodológica**, no predictiva:

- proporciona contexto económico al target definido en retornos,
- conecta el espacio estadístico del modelado con el espacio económico de evaluación,
- garantiza coherencia entre la formulación del problema y la interpretación de resultados.

## **Síntesis**

El análisis de `close` no persigue definir umbrales ni criterios operativos, sino **establecer una referencia de escala** que permita interpretar correctamente los retornos y sus errores asociados.  

Solo a partir de esta comprensión resulta metodológicamente consistente avanzar hacia el análisis estadístico detallado de los retornos y el entrenamiento de modelos predictivos.


# **2. Aplicación de análisis**

## **2.1 Análisis estadistico de `close`**


In [22]:
import pandas as pd
from typing import Dict, Tuple, Optional

def analyze_close_statistics(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    percentiles=(0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99),
    by_year: bool = True,
) -> Tuple[pd.Series, Dict[str, float], Optional[pd.DataFrame]]:
    """
    Calcula estadísticas descriptivas de la variable close (nivel del índice).

    Objetivo en este proyecto:
    - Entender la escala del precio (close) para interpretar retornos y convertirlos a puntos:
        ΔP ≈ r * close_t

    No modifica el índice del DataFrame.
    """

    # Validación suave: solo verificamos que exista la columna
    if close_col not in df.columns:
        raise ValueError(f"No existe la columna '{close_col}' en el DataFrame.")

    close_series = df[close_col].dropna()

    # 1) Estadísticas globales (incluye percentiles)
    close_describe = close_series.describe(percentiles=percentiles)

    # 2) Resumen compacto (para logging / reporte)
    summary = {
        "count": int(close_series.count()),
        "mean": float(close_series.mean()),
        "std": float(close_series.std()),
        "min": float(close_series.min()),
        "p01": float(close_series.quantile(0.01)),
        "p05": float(close_series.quantile(0.05)),
        "p50": float(close_series.quantile(0.50)),
        "p95": float(close_series.quantile(0.95)),
        "p99": float(close_series.quantile(0.99)),
        "max": float(close_series.max()),
    }

    # 3) Estadísticas por año (opcional)
    close_by_year = None
    if by_year:
        # Nota: no alteramos df.index; solo leemos el año
        if not isinstance(df.index, pd.DatetimeIndex):
            raise TypeError("El índice del DataFrame debe ser un DatetimeIndex para agrupar por año.")

        tmp = df[[close_col]].copy()
        tmp["year"] = tmp.index.year

        close_by_year = tmp.groupby("year")[close_col].agg(
            count="count",
            mean="mean",
            median="median",
            p05=lambda s: s.quantile(0.05),
            p95=lambda s: s.quantile(0.95),
            min="min",
            max="max",
        )

        # Métricas adicionales útiles para escala del precio
        close_by_year["range_p95_p05"] = close_by_year["p95"] - close_by_year["p05"]
        close_by_year["rel_range_p95_p05"] = close_by_year["range_p95_p05"] / close_by_year["median"]

    return close_describe, summary, close_by_year


## **2.2. Evaluación de resultados**

In [26]:
close_desc, close_summary, close_yearly = analyze_close_statistics(mnq_intraday, close_col="close", by_year=True)

In [25]:
print("\nclose_desc:\n")         # describe global
print(close_desc)         # describe global

print("\nclose_yearly:\n")         # describe global
display(close_yearly)     # tabla por año (más útil para régimen de precios)

print("\nclose_summary:\n")         # describe global
close_summary


close_desc:

count    744013.000000
mean      14762.034275
std        3566.973264
min        6765.750000
1%         8099.780000
5%         9110.500000
25%       12065.250000
50%       14430.750000
75%       17513.000000
95%       21284.000000
99%       21908.720000
max       22317.250000
Name: close, dtype: float64

close_yearly:



,count,mean,median,p05,p95,min,max,range_p95_p05,rel_range_p95_p05
year,,,,,,,,,
2019,2284,8755.341287,8735.000,8714.5375,8836.25,8695.25,8842.25,121.7125,0.013934
2020,129617,10279.788976,10374.250,7893.9000,12522.00,6765.75,12915.00,4628.1000,0.446114
2021,139895,14464.081150,14519.000,12858.0000,16346.75,12209.50,16756.00,3488.7500,0.240289
2022,137611,12800.716405,12478.750,11016.0000,15226.00,10495.75,16549.75,4210.0000,0.337374
2023,135327,14275.036959,14847.000,11610.5750,16255.75,10755.25,17162.50,4645.1750,0.312870
2024,136469,19177.319030,19031.250,17106.3500,21480.25,16337.00,22149.75,4373.9000,0.229827
2025,62810,20646.924132,21143.875,18405.0000,22021.50,16706.25,22317.25,3616.5000,0.171042



close_summary:



{'count': 744013,
 'mean': 14762.034274938744,
 'std': 3566.973263673845,
 'min': 6765.75,
 'p01': 8099.78,
 'p05': 9110.5,
 'p50': 14430.75,
 'p95': 21284.0,
 'p99': 21908.72,
 'max': 22317.25}

## **2.3. Comentarios sobre el nivel del índice (`close`)**

**Distribución global**

- El nivel del MNQ presenta una **amplia variabilidad histórica**, con valores que van desde ~6.800 hasta ~22.300 puntos.
- El rango interpercentil es muy amplio (p05 ≈ 9.100 vs p95 ≈ 21.284), confirmando la coexistencia de **múltiples regímenes de precio** dentro del dataset.
- La desviación estándar elevada en relación con la media refleja que el nivel del índice **no puede considerarse aproximadamente constante** a lo largo del período analizado.

---

**Evolución por año**

- Se observa un **cambio estructural claro en el nivel del índice** entre 2019 y 2025, con un crecimiento sostenido del precio.
- El año 2020 muestra el mayor rango relativo (`rel_range_p95_p05 ≈ 0.45`), consistente con un régimen de alta volatilidad.
- En años posteriores, aunque el nivel del índice es más alto, el rango relativo tiende a reducirse, indicando una **normalización gradual de la volatilidad relativa**.
- La mediana anual del índice se multiplica aproximadamente por 2,4 entre 2019 y 2025, confirmando la **no estacionariedad del nivel del precio**.

---

**Implicancias metodológicas**

- Dado que el mismo movimiento absoluto en puntos representa **retornos muy distintos** según el año, el uso de deltas en puntos como target introduciría una fuerte dependencia del régimen de precios.
- El análisis refuerza la conveniencia de utilizar **retornos como variable objetivo**, al ser invariantes al nivel del índice.
- La variable `close` debe conservarse como **referencia de escala** para la conversión de errores relativos a puntos, pero no como target directo.

---

**Síntesis**

El análisis del nivel del índice confirma que el dataset cubre múltiples regímenes de precios y que la escala del activo cambia sustancialmente a lo largo del tiempo. Este comportamiento valida la elección de los **retornos** como target de predicción y justifica la evaluación económica posterior en términos de puntos.

# **3. Evaluación de los retornos simples**


Una vez definido el **target de predicción** como el retorno simple y analizado el **nivel y la variabilidad del índice MNQ** a través de la variable `close`, corresponde evaluar el **comportamiento estadístico de los retornos intradía**.

El objetivo de esta sección es **caracterizar empíricamente la distribución de los retornos** y verificar si presentan propiedades compatibles con un enfoque predictivo, tales como:

- frecuencia suficiente de movimientos no triviales,
- estabilidad relativa entre distintos horizontes temporales,
- ausencia de una concentración excesiva alrededor de cero.

En esta etapa **no se imponen objetivos económicos ni umbrales operativos a priori**. El análisis se realiza de forma descriptiva y exploratoria, permitiendo que las conclusiones emerjan directamente de los datos.

---

**Retornos considerados**

Se analizan los **retornos simples acumulados** a los siguientes horizontes temporales:

- `ret_60`: retorno acumulado a 60 minutos  
- `ret_90`: retorno acumulado a 90 minutos  

Cada retorno se define como:

$$
r_{t,h} = \frac{close_{t+h} - close_t}{close_t}
$$

y representa la variación relativa del precio del índice MNQ en la ventana temporal correspondiente.

---

**Rol de esta sección dentro del estudio**

El análisis estadístico de los retornos cumple una función central dentro del pipeline:

- permite entender la **escala típica y la dispersión** del target,
- facilita la comparación entre horizontes temporales,
- y sienta las bases para evaluar la **frecuencia empírica de movimientos relevantes** sin introducir supuestos externos.

Solo a partir de esta caracterización resulta metodológicamente consistente avanzar hacia:
- el análisis de estabilidad temporal,
- la definición de estrategias de modelado,
- y la posterior interpretación económica de los resultados.

## **3.1. Cálculo de retornos por horizonte temporal**

In [27]:
import numpy as np
import pandas as pd

def compute_targets_returns(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    date_col: str = "date",
    horizons: tuple[int, ...] = (60, 90),
) -> pd.DataFrame:
    """
    Fórmulas:
      r_{t,h}  = (P_{t+h} - P_t) / P_t
    Sin cruzar días (shift por date). Índice datetime intacto.
    """
    out = df.copy()

    for h in horizons:
        P_t = out[close_col]
        P_th = out.groupby(date_col, group_keys=False)[close_col].shift(-h)

        out[f"ret_{h}"]   = (P_th - P_t) / P_t

    return out

In [30]:
mnq_intraday_targets = compute_targets_returns(
    mnq_intraday,
    close_col="close",
    date_col="date",
    horizons=(60, 90),
)

In [31]:
mnq_intraday_targets

,date,open,high,low,close,volume,ret_60,ret_90
datetime,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,0.001031,0.000687
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,0.001060,0.000859
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,0.001117,0.000917
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,0.000974,0.000917
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,0.000917,0.000888
...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,NaN,NaN
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,NaN,NaN
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,NaN,NaN


## **3.2. Análisis descriptivo de retornos simples por horizonte temporal**

### **3.2.1. Código de aplicación**

In [19]:
returns = ['ret_60', 'ret_90']

In [32]:
returns_stats = (
    mnq_intraday_targets[returns]
    .dropna()
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
)

### **3.2.2. Análisis y Conclusiones**

In [33]:
returns_stats

,ret_60,ret_90
count,626743.000000,626743.000000
mean,0.000060,0.000090
std,0.004267,0.005273
min,-0.054979,-0.048665
1%,-0.012334,-0.014923
5%,-0.006717,-0.008405
50%,0.000189,0.000244
95%,0.006166,0.007768
99%,0.011473,0.013938
max,0.083174,0.086742


### **3.2.3. Conclusiones - Distribución de los retornos simples (`ret_60` y `ret_90`)**

- Ambos retornos presentan una **media positiva muy cercana a cero**, consistente con un mercado intradía aproximadamente no sesgado en promedio.
- La **mediana positiva** en ambos horizontes indica una ligera asimetría hacia movimientos alcistas, aunque de magnitud reducida.
- Al aumentar el horizonte de 60 a 90 minutos, la **dispersión del retorno se incrementa** (`std` de 0.0043 a 0.0053), reflejando la acumulación natural de volatilidad.
- Las colas de la distribución se ensanchan con el horizonte:
  - el rango entre los percentiles extremos (p01-p99) es mayor en `ret_90` que en `ret_60`,
  - los valores máximos y mínimos absolutos aumentan de forma moderada.
- La mayor parte de los retornos se concentra en un intervalo estrecho alrededor de cero, lo que confirma que los **movimientos extremos son infrecuentes** en ambos horizontes.
- La similitud estructural entre `ret_60` y `ret_90` sugiere que el comportamiento del retorno es **estable entre horizontes**, diferenciándose principalmente por escala.

---

**Implicancia metodológica**

Los resultados indican que los retornos simples a 60 y 90 minutos poseen:
- una distribución bien definida,
- colas moderadas,
- y una variabilidad que crece de forma controlada con el horizonte.

Estas propiedades son compatibles con un enfoque predictivo y respaldan su uso como **target de modelado intradía**, permitiendo comparar horizontes sin introducir cambios estructurales en la definición del problema.


# **4. Vinculación entre nivel de precio (`close`), retornos y magnitud económica**


## **4.1. Explicación metodológica**

El objetivo de este punto es integrar los resultados obtenidos en el análisis del **nivel del índice MNQ** (Punto 2) y del **comportamiento estadístico de los retornos** (Punto 3), con el fin de establecer una **relación cuantitativa coherente** entre:

- el nivel del índice MNQ (`close`),
- los retornos simples intradía (`ret_h`),
- y la magnitud económica de los movimientos expresada en puntos del índice.

Esta vinculación no persigue definir nuevos targets ni criterios operativos, sino **proporcionar un marco de interpretación económica** que permita traducir retornos relativos a variaciones absolutas en puntos.


## **4.2 Principio de correspondencia entre retornos y puntos**


En este proyecto, el modelado se realiza sobre **retornos simples**, definidos como:

$$
r_{t,h} = \frac{close_{t+h} - close_t}{close_t}
$$

Sin embargo, la interpretación económica de los resultados se expresa naturalmente en **puntos del índice**. La relación entre ambos dominios viene dada por:

$$
\Delta P_{t,h} \approx r_{t,h} \cdot close_t
$$

donde:
- $ r_{t,h} $ es el retorno simple acumulado en el horizonte $h$,
- $ close_t $ es el nivel del índice al inicio de la ventana.

Esta expresión permite traducir cualquier retorno observado o predicho a una magnitud absoluta directamente interpretable desde el punto de vista económico.


## **4.3 `close` como factor de escala dinámico**


El análisis histórico del nivel del índice evidenció que `close` presenta **variaciones significativas a lo largo del tiempo**, cubriendo múltiples regímenes de precios.

En consecuencia:
- la conversión entre retornos y puntos **no puede basarse en un valor fijo de referencia**,
- cada observación debe evaluarse en función de su propio $ close_t $,
- el impacto económico de un mismo retorno relativo **depende del nivel del mercado** en ese instante.

Este enfoque garantiza que la interpretación económica:
- sea coherente entre distintos períodos históricos,
- no esté sesgada hacia regímenes de precios específicos,
- y refleje correctamente la escala real del movimiento del índice.

## **4.4 Relación con el horizonte temporal**


La magnitud económica asociada a un retorno depende también del **horizonte temporal** considerado, dado que:

- horizontes mayores tienden a acumular mayor dispersión,
- horizontes menores concentran movimientos más frecuentes pero de menor amplitud.

Por este motivo, la relación entre retorno, precio y puntos se analiza **en conjunto con el horizonte**, permitiendo comparar de forma consistente la escala económica de `ret_60`, `ret_90` y otros horizontes intradía.


## **4.5 Rol de esta vinculación dentro del pipeline**


Este punto cumple una función **interpretativa y de validación económica**, no predictiva:

- conecta el espacio estadístico del modelado (retornos) con el espacio económico (puntos),
- permite evaluar errores y resultados en unidades operativas reales,
- y asegura coherencia entre la formulación matemática del problema y su interpretación práctica.

## **Cierre del punto**


En síntesis, la vinculación entre el nivel del índice, los retornos simples y la magnitud económica de los movimientos permite interpretar los resultados del modelado en términos de puntos reales del MNQ, sin introducir supuestos adicionales ni sesgos de escala.  

Este marco asegura que el análisis estadístico y los modelos predictivos permanezcan alineados con la realidad económica del instrumento, preservando la capacidad de generalización temporal del enfoque adoptado.

# **5. Evaluación de la frecuencia empírica de movimientos económicamente relevantes (desde el objetivo hacia la frecuencia real)**





## **5.1. Marco teórico**

La pregunta central que se aborda en este punto es la siguiente:

**¿Es viable, desde una perspectiva empírica, el objetivo económico planteado?**

Hasta este punto del análisis se ha establecido:

- Cuántos puntos por operación son necesarios para cumplir los objetivos operativos (Punto 1).
- En qué rangos de precio opera el índice MNQ a lo largo del tiempo (Punto 2).
- Cómo se distribuyen los retornos intradía para distintos horizontes temporales (Punto 3).

Sin embargo, aún resta responder una cuestión fundamental: **con qué frecuencia real el mercado alcanza dichos objetivos**.

Este sub-punto tiene como finalidad responder a la siguiente pregunta clave:

**¿Con qué probabilidad histórica el MNQ se mueve al menos Δ puntos dentro de un horizonte intradía dado?**


#### **4.3.2. Cálculo de frecuencias**

Para cada horizonte $h \in \{30, 60, 90, 120\}$ y para cada objetivo expresado en puntos $\Delta$, se evalúa la siguiente condición:

$$
ret_{t,h} \ge \frac{\Delta}{close_t}
$$

La frecuencia empírica se define como el porcentaje de observaciones que cumplen dicha condición respecto del total de observaciones válidas.

Este análisis responde directamente a la pregunta:

**“¿En qué proporción de los casos el mercado se movió al menos $\Delta$ puntos en $h$ minutos?”**

In [37]:
import pandas as pd
import numpy as np

def compute_move_frequencies(
    df: pd.DataFrame,
    close_col: str = "close",
    return_cols = ("ret_60", "ret_90"),
    point_targets = (25, 62.5)
) -> pd.DataFrame:
    """
    Calcula la frecuencia empírica con la que el MNQ alcanza
    distintos objetivos en puntos para varios horizontes temporales.

    Parámetros
    ----------
    df : DataFrame
        Dataset con columna 'close' y retornos logarítmicos.
    close_col : str
        Nombre de la columna de precio.
    return_cols : tuple
        Columnas de retornos (ret_30, ret_60, etc.).
    point_targets : tuple
        Objetivos en puntos a evaluar (ej. 25, 62.5).

    Retorna
    -------
    DataFrame con frecuencias empíricas (%).
    """

    results = []

    for pts in point_targets:
        for ret_col in return_cols:
            mask = df[[close_col, ret_col]].dropna().index
            close_t = df.loc[mask, close_col]
            ret_t = df.loc[mask, ret_col]

            r_threshold = pts / close_t
            freq = (ret_t >= r_threshold).mean()

            results.append({
                "horizon": ret_col,
                "points_target": pts,
                "frequency": freq
            })

    return pd.DataFrame(results)

In [38]:
freq_df = compute_move_frequencies(
    mnq_intraday_targets,
    point_targets=(25, 62.5)
)

#### **4.3.3. Resultados y análisis**

In [39]:
freq_df

,horizon,points_target,frequency
0,ret_60,25.0,0.276945
1,ret_90,25.0,0.324832
2,ret_60,62.5,0.095093
3,ret_90,62.5,0.137785


**1. Comportamiento consistente con el horizonte temporal**

  Los resultados muestran un patrón monótono y coherente: para ambos objetivos en puntos (25 y 62.5), la frecuencia de cumplimiento aumenta sistemáticamente al ampliar el horizonte temporal de análisis.

  Esto confirma que:
  - la metodología de cálculo es correcta,
  - los retornos están correctamente alineados con el horizonte temporal,
  - ventanas más largas permiten capturar movimientos de mayor magnitud con mayor probabilidad.

<br>

**2. Viabilidad relativa del objetivo mínimo (25 puntos)**

El objetivo de 25 puntos presenta las siguientes frecuencias empíricas:
- 30 minutos: ~19 %
- 60 minutos: ~27 %
- 90 minutos: ~32 %
- 120 minutos: ~36 %

Estas cifras indican que:
- el movimiento ocurre con una frecuencia no despreciable,
- especialmente en horizontes de 60 a 90 minutos, donde se aproxima a 1 de cada 3 observaciones,
- el objetivo resulta estadísticamente viable, aunque no trivial.

Desde el punto de vista operativo, esto sugiere que el objetivo mínimo puede ser alcanzado de forma consistente bajo un esquema selectivo, no aleatorio.
<br><br>
**3. Baja frecuencia del objetivo ideal (62.5 puntos)**

El objetivo de 62.5 puntos muestra frecuencias sensiblemente menores:
- 30 minutos: ~4.6 %
- 60 minutos: ~9.4 %
- 90 minutos: ~13.7 %
- 120 minutos: ~17.5 %

Estos valores evidencian que:
- el movimiento existe, pero ocurre con baja frecuencia,
- incluso en horizontes extendidos, se mantiene por debajo del 20 %,
- corresponde a escenarios excepcionales, más asociados a impulsos direccionales fuertes.

Esto descarta su uso como objetivo base para una operatoria diaria sistemática.
<br><br>
**4. Diferenciación clara entre objetivo base y extensión**

El contraste entre ambos objetivos permite establecer una separación natural:
- 25 puntos → movimiento relativamente frecuente y operable.
- 62.5 puntos → movimiento de baja probabilidad, adecuado como extensión condicional.

Esta diferenciación emerge de los datos, no de supuestos externos.
<br><br>
**5. Implicancia directa sobre la planificación operativa**

Los resultados indican que:
- aumentar el horizonte temporal incrementa la probabilidad de alcanzar los objetivos,
- pero no de manera proporcional para objetivos ambiciosos.

En consecuencia:
- el incremento del horizonte no compensa completamente la baja frecuencia de movimientos extensos,
- la planificación debe priorizar objetivos alineados con la zona de mayor densidad probabilística.
<br><br>
**Conclusiones del Punto 4.3**

1. La frecuencia empírica es un criterio indispensable para evaluar la viabilidad real de los objetivos operativos.
2. El objetivo mínimo de 25 puntos es estadísticamente defendible, especialmente en horizontes de 60 a 90 minutos.
3. El objetivo ideal de 62.5 puntos presenta una probabilidad baja y no debe considerarse como expectativa base.
4. La diferencia de frecuencias justifica un enfoque operativo basado en:
    - objetivos frecuentes como núcleo,
    - extensiones ocasionales como complemento.
5. Estos resultados proporcionan una base objetiva para avanzar hacia la definición de targets, evitando decisiones arbitrarias.

### **4.4. Análisis empírico inverso del movimiento intradía en puntos**

#### **4.4.1. Marco conceptual**

##### **1. Motivación y propósito del análisis**

Hasta este punto del trabajo, la vinculación entre retornos, nivel de precio y objetivos económicos se ha abordado desde un enfoque top-down, partiendo de metas operativas previamente definidas (en puntos y dólares) y evaluando su viabilidad estadística a través de la frecuencia empírica de cumplimiento (Puntos 4.2 y 4.3).

Si bien dicho enfoque permite validar si un objetivo es razonable o no, aún persiste una cuestión fundamental:

**¿Qué magnitudes de movimiento intradía genera naturalmente el mercado, independientemente de los objetivos económicos propuestos?**

Con el fin de responder a esta pregunta y reforzar la solidez metodológica del análisis, se introduce en este punto un enfoque inverso o bottom-up, en el cual se deja que el comportamiento histórico del MNQ determine las magnitudes de movimiento más frecuentes.

##### **2. Enfoque metodológico**

El análisis empírico inverso consiste en estudiar directamente la magnitud efectiva de los movimientos intradía, expresada en puntos del índice, para distintos horizontes temporales.

Para ello, los retornos logarítmicos calculados previamente se transforman en variaciones absolutas en puntos mediante la relación:

  $$
    \Delta \text{puntos}_{t,h} \approx r_{t,h} \times close_t
  $$
      


Este procedimiento permite:
- abandonar temporalmente la referencia a objetivos prefijados,
- analizar el movimiento real del mercado en unidades directamente operativas,
- observar la distribución completa de los desplazamientos intradía.


##### **3. Qué se busca identificar**

A través de este análisis se busca:

- identificar la magnitud típica del movimiento intradía del MNQ para cada horizonte temporal,
- detectar rangos de puntos que concentran la mayor densidad probabilística,
- distinguir entre:
  - movimientos frecuentes y estructurales del mercado,
  - movimientos excepcionales asociados a colas de la distribución.

En particular, el foco no está puesto en los valores extremos, sino en aquellos desplazamientos que ocurren de manera recurrente y estable, y que por lo tanto constituyen una base más sólida para el diseño de una estrategia operativa sistemática.

##### **4. Rol de este análisis dentro del proceso de definición de targets**

El análisis empírico inverso no reemplaza a los objetivos económicos definidos inicialmente, sino que los complementa y valida desde la perspectiva del comportamiento real del mercado.

Su rol dentro del pipeline es:
- evitar la imposición de umbrales arbitrarios,
- permitir que los datos históricos del MNQ dicten las magnitudes de movimiento más frecuentes,
- proporcionar un criterio adicional, independiente y empírico, para la posterior definición de los targets de predicción.

##### **5. Cierre conceptual**

En síntesis, este sub-punto introduce una validación fundamental: los objetivos operativos no solo deben ser económicamente deseables y estadísticamente viables, sino también consistentes con la dinámica intradía real del mercado. Solo a partir de esta doble validación resulta posible avanzar hacia una definición de targets que sea robusta, defendible y alineada con una operatoria sostenible en el tiempo.

#### **4.4.2. Código de aplicación y cálculo**

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 4.4 — Análisis empírico inverso del movimiento intradía en puntos
# ============================================================
# Objetivo:
# 1) Convertir retornos logarítmicos (ret_h) a movimientos en puntos (Δpts_h)
# 2) Resumir la distribución de |Δpts_h| (magnitud) por horizonte
# 3) Calcular frecuencias del tipo P(|Δpts_h| >= umbral)
# 4) Estimar el "delta más frecuente" (modo aproximado) usando bins/histograma
#
# Nota clave:
# - "frecuencia >= umbral" responde a "al menos X puntos".
# - "modo" (más repetido) requiere histograma/bins, no umbrales.
# ============================================================


def compute_points_moves(
    df: pd.DataFrame,
    close_col: str = "close",
    return_cols=("ret_30", "ret_60", "ret_90", "ret_120"),
    prefix: str = "pts_",
    method: str = "approx",   # "approx" o "exact"
) -> pd.DataFrame:
    """
    Crea columnas de movimiento en puntos por horizonte:
      - approx:  Δpts ≈ ret_h * close_t
      - exact:   Δpts ≈ close_t * (exp(ret_h) - 1)

    ¿Cuál usar?
    - approx es muy buena para movimientos intradía pequeños.
    - exact es más fiel para colas (eventos grandes).
    """
    if method not in ("approx", "exact"):
        raise ValueError("method debe ser 'approx' o 'exact'.")

    out = df.copy()

    for rc in return_cols:
        h = rc.replace("ret_", "")  # "30", "60", "90", "120"
        if method == "exact":
            # Δpts = close_t * (e^{ret} - 1)
            out[f"{prefix}{h}"] = out[close_col] * (np.exp(out[rc]) - 1.0)
        else:
            # Aproximación lineal: Δpts ≈ ret * close
            out[f"{prefix}{h}"] = out[rc] * out[close_col]

    return out


def summarize_points_moves(
    df: pd.DataFrame,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    percentiles=(0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99),
    use_abs: bool = True
) -> pd.DataFrame:
    """
    Resume la distribución de movimientos en puntos por horizonte.

    use_abs=True  -> analiza magnitud |Δpts| (útil para objetivos económicos sin dirección).
    use_abs=False -> analiza Δpts con signo (útil si se quiere separar long/short).
    """
    data = df.loc[:, points_cols].copy()
    if use_abs:
        data = data.abs()

    summary = data.describe(percentiles=percentiles).T

    # Reorden de columnas para lectura
    cols_order = ["count", "mean", "std", "min"] + [f"{int(p*100)}%" for p in percentiles] + ["max"]
    summary = summary[[c for c in cols_order if c in summary.columns]]
    return summary


def points_threshold_frequencies(
    df: pd.DataFrame,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    thresholds=(10, 15, 20, 25, 30, 40, 50, 62.5, 75, 100),
    use_abs: bool = True
) -> pd.DataFrame:
    """
    Calcula frecuencias empíricas del tipo:
        P(|Δpts_h| >= umbral)

    IMPORTANTE:
    - Esto responde a "¿con qué frecuencia se mueve AL MENOS X puntos?".
    - NO responde a "¿cuál es el delta más repetido?" (eso se hace con histograma/modo).
    """
    data = df.loc[:, points_cols].copy()
    if use_abs:
        data = data.abs()

    rows = []
    for col in points_cols:
        s = data[col].dropna()
        for thr in thresholds:
            rows.append({
                "horizon_pts": col,
                "points_threshold": float(thr),
                "frequency": float((s >= thr).mean()),
                "n": int(s.shape[0])
            })

    return pd.DataFrame(rows)


def mode_points_by_bins(
    df: pd.DataFrame,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    bin_width: float = 5.0,
    max_abs_points: float = 300.0,
    use_abs: bool = True
) -> pd.DataFrame:
    """
    Estima el "delta más frecuente" (modo aproximado) usando histograma por bins.

    - bin_width: ancho del bin en puntos (ej. 5 puntos).
    - max_abs_points: recorta a [0, max_abs_points] (o [-max, +max si use_abs=False)
      para evitar que colas extremas dominen la escala del histograma.

    Retorna para cada horizonte:
    - bin_left, bin_right: límites del bin modal
    - mode_midpoint: punto medio del bin modal (estimación del modo)
    - bin_frequency: frecuencia del bin modal (aprox densidad)
    - n_used: cantidad de observaciones consideradas
    """
    rows = []
    for col in points_cols:
        s = df[col].dropna()

        # Usualmente interesa magnitud (sin dirección)
        if use_abs:
            s = s.abs()
            s = s[(s >= 0) & (s <= max_abs_points)]
            bins = np.arange(0, max_abs_points + bin_width, bin_width)
        else:
            s = s[(s >= -max_abs_points) & (s <= max_abs_points)]
            bins = np.arange(-max_abs_points, max_abs_points + bin_width, bin_width)

        if s.empty:
            rows.append({
                "horizon_pts": col,
                "bin_left": np.nan,
                "bin_right": np.nan,
                "mode_midpoint": np.nan,
                "bin_frequency": np.nan,
                "n_used": 0
            })
            continue

        # Histograma: counts por bin
        counts, edges = np.histogram(s.values, bins=bins)
        idx = int(np.argmax(counts))

        left = float(edges[idx])
        right = float(edges[idx + 1])
        midpoint = (left + right) / 2.0
        freq = float(counts[idx] / counts.sum())

        rows.append({
            "horizon_pts": col,
            "bin_left": left,
            "bin_right": right,
            "mode_midpoint": midpoint,
            "bin_frequency": freq,
            "n_used": int(s.shape[0])
        })

    return pd.DataFrame(rows)


In [ ]:
# (A) Dataset con retornos: mnq_intraday_with_returns
#     - Debe contener: close y ret_30/ret_60/ret_90/ret_120

# 1) Convertir a puntos (elige método):
#    - "approx" (rápido, suficiente para intradía típico)
#    - "exact"  (más fiel para colas)
mnq_with_pts = compute_points_moves(
    mnq_intraday_with_returns,
    method="approx"  # o "exact"
)

# 2) Resumen por percentiles (magnitud en puntos)
pts_summary = summarize_points_moves(
    mnq_with_pts,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    use_abs=True
)
# 3) Frecuencias "al menos X puntos"
freq_pts = points_threshold_frequencies(
    mnq_with_pts,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    thresholds=(10, 15, 20, 25, 30, 40, 50, 62.5, 75, 100),
    use_abs=True
)
# 4) "Delta más frecuente" (modo aproximado) por bins
#    - bin_width=5 significa "modo por intervalos de 5 puntos"
modes = mode_points_by_bins(
    mnq_with_pts,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    bin_width=5.0,
    max_abs_points=300.0,
    use_abs=True
)

#### **4.4.3. Distribución empírica de |Δpuntos| por horizonte temporal**

Este primer resultado presenta la distribución de la magnitud absoluta del movimiento intradía, expresada en puntos del índice MNQ, para distintos horizontes temporales (30, 60, 90 y 120 minutos).

In [ ]:
print("=== Distribución |Δpts| por horizonte ===\n")
pts_summary

=== Distribución |Δpts| por horizonte ===



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
pts_30,704923.0,27.445000,31.298248,0.0,0.250002,1.500070,7.997591,18.019786,35.950696,84.686966,143.620227,1381.681382
pts_60,665833.0,40.001235,44.317640,0.0,0.499992,2.249885,11.753815,26.782521,52.894542,121.492043,204.367265,1399.090763
pts_90,626743.0,50.299615,54.376302,0.0,0.500009,2.750274,14.988925,34.212038,67.659027,150.406684,249.487806,1460.053659
pts_120,587653.0,59.889531,62.883501,0.0,0.749981,3.500311,18.237892,41.544095,81.337567,175.428589,284.574905,1576.307287


**1. Comportamiento general de la distribución**

Se observa un patrón consistente y esperable en todos los horizontes:
- El movimiento promedio (mean) aumenta de manera monótona con el horizonte temporal.
- El desvío estándar crece de forma proporcional, reflejando mayor dispersión a medida que se amplía la ventana temporal.
- La distribución presenta colas largas, evidenciadas por valores máximos muy elevados.

Este comportamiento confirma que:
- el procedimiento de conversión de retornos a puntos es correcto,
- el mercado acumula mayor desplazamiento cuanto mayor es el horizonte analizado.
<br><br>

**2. Movimiento típico del mercado (mediana)**

La mediana (50 %) representa el movimiento más representativo del mercado en cada horizonte:
- 30 min: ~18 puntos
- 60 min: ~27 puntos
- 90 min: ~34 puntos
- 120 min: ~42 puntos

Interpretación clave:

Estos valores describen el movimiento intradía típico del MNQ y constituyen una referencia empírica directa de la magnitud que el mercado genera de forma recurrente.
<br><br>

**3. Zona de alta densidad (rango intercuartílico)**

El rango P25–P75 define la región donde se concentra el 50 % central de las observaciones:
- 30 min: ~8 a ~36 puntos
- 60 min: ~12 a ~53 puntos
- 90 min: ~15 a ~68 puntos
- 120 min: ~18 a ~81 puntos

Esto indica que:
- en horizontes de 60 a 90 minutos, la mayor parte de los movimientos intradía se ubican entre 20 y 60 puntos,
- este rango coincide con la zona de mayor interés operativo desde el punto de vista económico.
<br><br>

**4. Movimientos grandes y colas de la distribución**

Los percentiles altos muestran la presencia de movimientos extensos:
- P95: entre ~85 y ~175 puntos, según el horizonte.
- P99: entre ~144 y ~285 puntos.

Estos valores corresponden a:
- eventos poco frecuentes,
- impulsos direccionales fuertes,
- situaciones de alta volatilidad.

Conclusión:

Estos movimientos no deben utilizarse como referencia para objetivos operativos base, pero confirman que el dataset captura correctamente escenarios extremos del mercado.
<br><br>

**5. Implicancias directas para la operatoria**

A partir de esta distribución se desprenden varias conclusiones clave:

1. El MNQ presenta un movimiento intradía típico bien definido, que crece con el horizonte temporal.
2. La zona de mayor densidad probabilística se encuentra en el rango de 20–40 puntos para horizontes de 60–90 minutos.
3. Movimientos superiores a 60 puntos pertenecen progresivamente a la cola de la distribución y deben considerarse como extensiones.

<br><br>
**Conclusión parcial del Punto 4.4.3**

El análisis de la distribución empírica de |Δpuntos| muestra que el mercado genera, de manera natural y recurrente, movimientos intradía de magnitud moderada, claramente identificables y estables en el tiempo. Estos resultados proporcionan una base objetiva y empírica para la posterior selección de targets alineados con la dinámica real del MNQ.

#### **4.4.4.Frecuencia empírica de movimientos en puntos por horizonte**

Este bloque evalúa la probabilidad histórica de que el MNQ registre un movimiento intradía cuya magnitud absoluta sea al menos un determinado umbral de puntos, para distintos horizontes temporales.

In [ ]:
print("=== Frecuencias P(|Δpts| >= umbral) ===\n")
freq_pts.sort_values(["horizon_pts", "points_threshold"])

=== Frecuencias P(|Δpts| >= umbral) ===



,horizon_pts,points_threshold,frequency,n
30,pts_120,10.0,0.857962,587653
31,pts_120,15.0,0.791021,587653
32,pts_120,20.0,0.727329,587653
33,pts_120,25.0,0.668252,587653
34,pts_120,30.0,0.613148,587653
35,pts_120,40.0,0.513623,587653
36,pts_120,50.0,0.430601,587653
37,pts_120,62.5,0.345619,587653
38,pts_120,75.0,0.278984,587653
39,pts_120,100.0,0.182586,587653


**1. Patrón general de las frecuencias**

Se observa un comportamiento monótono y consistente en todos los horizontes:
- A medida que el umbral en puntos aumenta, la frecuencia de ocurrencia disminuye.
- Para un mismo umbral, la frecuencia crece con el horizonte temporal.

Este patrón valida:
- la coherencia del cálculo,
- la relación directa entre tiempo disponible y probabilidad de capturar movimientos de mayor magnitud.

<br>

**2. Umbrales de alta probabilidad (movimientos “casi seguros”)**

Considerando umbrales que ocurren en aproximadamente 70–80 % de los casos:
- 30 min: ≥ 10–15 puntos
- 60 min: ≥ 15–20 puntos
- 90 min: ≥ 15–20 puntos
- 120 min: ≥ 20–25 puntos

Interpretación:

Estos valores representan los movimientos intradía más frecuentes y estables, constituyendo el límite superior de lo que puede considerarse “alta probabilidad” en cada horizonte.

<br>

**3. Umbrales de probabilidad intermedia (zona operativa)**

Para frecuencias en el rango 30–60 %, se identifica una zona operativa natural:
- 30 min: ~20–30 puntos
- 60 min: ~25–40 puntos
- 90 min: ~25–40 puntos
- 120 min: ~30–50 puntos

Esta región coincide con:
- la mediana y el rango intercuartílico observados en la distribución de |Δpuntos|,
- los objetivos económicos mínimos planteados previamente.

<br>

**4. Movimientos de baja frecuencia (extensiones)**

Umbrales con frecuencia inferior al 20 % corresponden a:
- 30 min: ≥ 50–62.5 puntos
- 60 min: ≥ 62.5–75 puntos
- 90 min: ≥ 75–100 puntos
- 120 min: ≥ 75–100 puntos

Conclusión:

Estos movimientos existen, pero pertenecen a la cola de la distribución y deben considerarse como escenarios de extensión, no como expectativa base.

<br>

**5. Comparación entre horizontes (lectura transversal)**

- 30 minutos: alta rotación, pero limitada capacidad de capturar movimientos grandes.
- 60 minutos: equilibrio óptimo entre frecuencia y magnitud.
- 90 minutos: mayor probabilidad de capturar extensiones con una frecuencia aún razonable.
- 120 minutos: máxima probabilidad para movimientos grandes, a costa de menor flexibilidad intradía.

<br>

**6. Implicancias para la definición de objetivos**

Este análisis permite establecer criterios claros:

1. Los objetivos con alta frecuencia garantizan consistencia pero menor ganancia por operación.
2. Los objetivos con frecuencia intermedia representan el mejor compromiso riesgo–retorno.
3. Los objetivos de baja frecuencia deben reservarse como extensiones condicionadas a contexto favorable.

<br>

**Conclusión parcial del Punto 4.4.4**

La frecuencia empírica de movimientos en puntos confirma que el MNQ presenta una estructura de desplazamientos intradía predecible en términos probabilísticos. Existen rangos de puntos claramente diferenciados según su frecuencia, lo que permite fundamentar la selección de objetivos operativos alineados con la dinámica real del mercado y el horizonte temporal elegido.

#### **4.4.5. Identificación del movimiento modal (delta más frecuente)**

Este resultado identifica el bin modal de la distribución de |Δpuntos|, es decir, el intervalo de magnitud que ocurre con mayor frecuencia para cada horizonte temporal.

In [ ]:
print("=== Modo aproximado (bin modal) de |Δpts| ===\n")
modes.sort_values("horizon_pts")

=== Modo aproximado (bin modal) de |Δpts| ===



,horizon_pts,bin_left,bin_right,mode_midpoint,bin_frequency,n_used
3,pts_120,0.0,5.0,2.5,0.071970,582975
0,pts_30,0.0,5.0,2.5,0.161811,704507
1,pts_60,0.0,5.0,2.5,0.110763,664419
2,pts_90,0.0,5.0,2.5,0.088395,623812


**1. Resultado observado**

En todos los horizontes analizados, el bin modal corresponde a:
- Intervalo: 0–5 puntos
- Modo aproximado: 2.5 puntos

Frecuencia del bin modal:
- 30 min: ~16.2 %
- 60 min: ~11.1 %
- 90 min: ~8.8 %
- 120 min: ~7.2 %

<br>

**2. Interpretación correcta del resultado**

Este resultado **no indica** que el objetivo operativo deba ser 2.5 puntos.
Lo que indica es que: El movimiento intradía más frecuente del MNQ, medido minuto a minuto, es pequeño y corresponde a fluctuaciones de baja magnitud.

Esto es esperable en mercados financieros intradía:
- la mayor parte del tiempo el precio oscila dentro de rangos reducidos,
- los movimientos grandes se construyen por acumulación de micro-movimientos.

<br>

**3. Por qué el modo no es un buen target operativo**

Desde el punto de vista estadístico:
- el modo identifica el valor más frecuente,
- pero no necesariamente el más útil.

Desde el punto de vista operativo:
- movimientos de 0–5 puntos:
  - no cubren costos,
  - no compensan riesgo,
  - no permiten capturar el valor económico buscado.

Conclusión clave:
- El delta modal describe el “ruido” del mercado, no la oportunidad.

<br>

**4. Relación con los análisis anteriores**

Este resultado debe interpretarse en conjunto, no de forma aislada:

- Distribución (punto 4.4.3):
  - el movimiento típico (mediana) es mucho mayor (18–42 pts).

- Frecuencias por umbral (punto 4.4.4):
  - 20–40 pts ocurren con frecuencia relevante.

- Modo (punto 4.4.5):
  - 0–5 pts es el movimiento más común, pero de bajo valor económico.

Esto refuerza una idea central: **La estrategia no debe buscar el movimiento más frecuente, sino el movimiento frecuente y económicamente significativo.**

<br>

**5. Implicancia metodológica importante**

El análisis del modo cumple un rol clave:
- descarta explícitamente el uso del modo como criterio de target,
- evita una interpretación errónea del concepto de “lo que más se repite”,
- justifica por qué se utilizan percentiles y frecuencias acumuladas como criterios principales.

En otras palabras: **el modo explica qué hace el mercado la mayor parte del tiempo, pero los percentiles explican dónde está el valor operativo.**

<br>

**Conclusión parcial del Punto 4.4.5**

El movimiento modal del MNQ en todos los horizontes corresponde a oscilaciones de muy baja magnitud (0–5 puntos), asociadas al ruido intradía. Si bien este resultado es estadísticamente correcto, carece de relevancia económica directa. La identificación de este comportamiento refuerza la necesidad de definir objetivos operativos basados en movimientos menos frecuentes, pero suficientemente grandes como para justificar el riesgo asumido.

#### **4.4.6. Cierre de punto 4.4. (síntesis final)**

Combinando los tres bloques analizados:

1. El mercado presenta un movimiento típico de 20–40 puntos en 60–90 minutos.
2. Dichos movimientos ocurren con frecuencia suficiente para una operatoria sistemática.
3. El movimiento más frecuente (modo) es pequeño y no operable.

**Conclusión final:**

Los targets deben ubicarse por encima del ruido modal, dentro de la zona de alta densidad probabilística y relevancia económica, criterio que surge directamente del comportamiento empírico del mercado.

# **6. Análisis de `mnq_intraday_labeled`**


### **6.1. Objetivo del análisis:**

Queremos responder, con datos:

- En qué minutos del día el dataset genera más señales (`trade_60`, `trade_90`)
- Qué magnitud tienen los movimientos futuros (`delta_pts_60`, `delta_pts_90`) en esos minutos
- Qué tan “económicas” son esas señales (por ejemplo, % con `|delta| >= 25 pts`)
- Si hay persistencia: cuándo `|delta_90|` tiende a ser mayor que `|delta_60|`

### **6.2. Validaciones rápidas del dataset**

In [ ]:
import pandas as pd
import numpy as np

df = mnq_intraday_labeled.copy()

#display(df.head())
print(f'df.columns: {df.columns}\n')

print(f'df.index.dtype: {df.index.dtype}\n')

# Validación: el índice debe ser DatetimeIndex
assert isinstance(df.index, pd.DatetimeIndex), "El índice debe ser DatetimeIndex (datetime)."

# Validación: columnas mínimas
required = ["trade_60","hold_60","delta_pts_60","trade_90","hold_90","delta_pts_90"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Faltan columnas: {missing}"

# Recuento básico de señales
print("Trades 60:", (df["trade_60"] != 0).sum())
print("Trades 90:", (df["trade_90"] != 0).sum())

df.columns: Index(['date', 'open', 'high', 'low', 'close', 'volume', 'delta_pts_60',
       'trade_60', 'hold_60', 'delta_pts_90', 'trade_90', 'hold_90'],
      dtype='object')

df.index.dtype: datetime64[ns, America/New_York]

Trades 60: 331500
Trades 90: 382455


### **6.3 Feature auxiliar: minuto del día (para agrupar intradía)**

In [ ]:
df["minute_of_day"] = df.index.hour * 60 + df.index.minute
df["hour"] = df.index.hour
df["minute"] = df.index.minute

df[["minute_of_day","hour","minute"]].head()

,minute_of_day,hour,minute
datetime,,,
2019-12-23 06:30:00-05:00,390,6,30
2019-12-23 06:31:00-05:00,391,6,31
2019-12-23 06:32:00-05:00,392,6,32
2019-12-23 06:33:00-05:00,393,6,33
2019-12-23 06:34:00-05:00,394,6,34


### **6.4 Función principal: métricas por minuto del día**

Esta función calcula exactamente:

- Frecuencia: `n_trade`, `n_long`, `n_short` (para 60 y 90)
- Magnitud sobre `|delta_pts|` para trades: mean, p50, p75, p90 (para 60 y 90)
- Calidad: `%(|delta| >= thr_pts)` (para 60 y 90)
- Persistencia: `|delta_90| / |delta_60|` (stats + n_persist)

In [ ]:
def metrics_by_minute_of_day(
    df: pd.DataFrame,
    thr_pts: float = 25.0,
    trade_60="trade_60",
    trade_90="trade_90",
    d60="delta_pts_60",
    d90="delta_pts_90",
) -> pd.DataFrame:

    dfx = df[[trade_60, trade_90, d60, d90, "minute_of_day"]].copy()
    dfx["abs60"] = dfx[d60].abs()
    dfx["abs90"] = dfx[d90].abs()

    # Persistencia: solo cuando hay trade en ambos y abs60>0
    mask_p = (dfx[trade_60] != 0) & (dfx[trade_90] != 0) & (dfx["abs60"] > 0)
    dfx["persist_ratio"] = np.nan
    dfx.loc[mask_p, "persist_ratio"] = dfx.loc[mask_p, "abs90"] / dfx.loc[mask_p, "abs60"]

    def stats(x: pd.Series) -> pd.Series:
        x = x.dropna()
        if x.empty:
            return pd.Series({"mean": np.nan, "p50": np.nan, "p75": np.nan, "p90": np.nan})
        return pd.Series({
            "mean": float(x.mean()),
            "p50": float(x.quantile(0.50)),
            "p75": float(x.quantile(0.75)),
            "p90": float(x.quantile(0.90)),
        })

    def pct_ge(x: pd.Series, thr: float) -> float:
        x = x.dropna()
        return np.nan if x.empty else float((x >= thr).mean())

    g = dfx.groupby("minute_of_day", sort=True)
    out = pd.DataFrame(index=g.size().index)

    # Frecuencia 60
    out["n_trade_60"] = g.apply(lambda s: int((s[trade_60] != 0).sum()))
    out["n_long_60"]  = g.apply(lambda s: int((s[trade_60] == 1).sum()))
    out["n_short_60"] = g.apply(lambda s: int((s[trade_60] == -1).sum()))

    # Frecuencia 90
    out["n_trade_90"] = g.apply(lambda s: int((s[trade_90] != 0).sum()))
    out["n_long_90"]  = g.apply(lambda s: int((s[trade_90] == 1).sum()))
    out["n_short_90"] = g.apply(lambda s: int((s[trade_90] == -1).sum()))

    # Magnitud |delta| para trades (no long/short por ahora, para mantenerlo simple)
    tmp60 = g.apply(lambda s: stats(s.loc[s[trade_60] != 0, "abs60"]))
    tmp60.columns = [f"abs60_trade_{c}" for c in tmp60.columns]
    out = out.join(tmp60)

    tmp90 = g.apply(lambda s: stats(s.loc[s[trade_90] != 0, "abs90"]))
    tmp90.columns = [f"abs90_trade_{c}" for c in tmp90.columns]
    out = out.join(tmp90)

    # Calidad: %(|delta| >= thr_pts) para trades
    out["pct_abs60_ge_thr_trade"] = g.apply(lambda s: pct_ge(s.loc[s[trade_60]!=0, "abs60"], thr_pts))
    out["pct_abs90_ge_thr_trade"] = g.apply(lambda s: pct_ge(s.loc[s[trade_90]!=0, "abs90"], thr_pts))

    # Persistencia stats
    out["n_persist"] = g.apply(lambda s: int(s["persist_ratio"].notna().sum()))
    tmpp = g.apply(lambda s: stats(s["persist_ratio"]))
    tmpp.columns = [f"persist_ratio_{c}" for c in tmpp.columns]
    out = out.join(tmpp)

    # Hora/minuto legible
    out["hour"] = out.index // 60
    out["minute"] = out.index % 60
    out = out[["hour","minute"] + [c for c in out.columns if c not in ("hour","minute")]]

    return out


### **6.5. Ejecutar el análisis**

In [ ]:
metrics_minute = metrics_by_minute_of_day(df, thr_pts=25.0)
#display(metrics_minute.head(10))

### **6.6 Tablas “Top” para interpretar rápido**

#### 6.6.1. Top 20 por frecuencia (60 min)

In [ ]:
top60 = metrics_minute.sort_values("n_trade_60", ascending=False).head(20)
display(top60[["hour","minute","n_trade_60","abs60_trade_mean","abs60_trade_p50","abs60_trade_p90","pct_abs60_ge_thr_trade"]])

,hour,minute,n_trade_60,abs60_trade_mean,abs60_trade_p50,abs60_trade_p90,pct_abs60_ge_thr_trade
minute_of_day,,,,,,,
564,9,24,977,85.758956,70.500,156.750,1.0
570,9,30,975,87.781026,75.250,157.750,1.0
560,9,20,972,84.711420,68.500,153.500,1.0
569,9,29,970,88.200000,75.000,160.025,1.0
561,9,21,969,85.479360,70.500,154.050,1.0
565,9,25,966,87.010352,74.750,158.625,1.0
558,9,18,966,84.859731,68.250,152.125,1.0
571,9,31,966,86.954710,73.500,159.750,1.0
576,9,36,965,82.676425,69.000,151.650,1.0


**Comentarios:**

- Concentración temporal clara: los Top 20 se agrupan entre 09:16 y 09:36, indicando una ventana horaria con alta recurrencia operativa.

- Frecuencia elevada y estable: `n_trade_60` ≈ 950–977 por minuto del día, lo que sugiere consistencia estadística (no son outliers esporádicos).

- Magnitud del movimiento relevante: `abs60_trade_mean` en el rango ~83–88 pts; medianas ~69–75 pts y p90 ~151–160 pts, lo que confirma colas derechas significativas.

- Umbral siempre alcanzado: `pct_abs60_ge_thr_trade` = 1.0 en todos los casos → 100% de los trades superan el umbral definido; el criterio es poco discriminante en este bloque horario.

- Implicación directa: este tramo horario es prioritario para análisis y modelado; conviene endurecer el umbral o introducir filtros adicionales si se busca selectividad.

#### 6.6.2. Top 20 por frecuencia (90 min)

In [ ]:
top90 = metrics_minute.sort_values("n_trade_90", ascending=False).head(20)
display(top90[["hour","minute","n_trade_90","abs90_trade_mean","abs90_trade_p50","abs90_trade_p90","pct_abs90_ge_thr_trade"]])


,hour,minute,n_trade_90,abs90_trade_mean,abs90_trade_p50,abs90_trade_p90,pct_abs90_ge_thr_trade
minute_of_day,,,,,,,
563,9,23,1015,96.606650,81.250,180.750,1.0
565,9,25,1012,96.886117,80.750,178.700,1.0
555,9,15,1010,94.677228,79.875,172.325,1.0
566,9,26,1009,97.409068,82.000,179.100,1.0
564,9,24,1007,97.415591,80.750,181.750,1.0
561,9,21,1007,96.598312,81.500,178.600,1.0
562,9,22,1007,96.402929,80.250,176.850,1.0
553,9,13,1007,94.071251,77.500,174.600,1.0
556,9,16,1007,95.075968,80.000,174.950,1.0


**Comentarios:**

- Ventana horaria consistente: nuevamente se concentra entre 09:11 y 09:36, reforzando que el núcleo operativo es el mismo que en 60 min.
- Mayor frecuencia: n_trade_90 ≈ 999–1015, superior a 60 min → más oportunidades acumuladas por minuto del día.
- Mayor magnitud esperada: abs90_trade_mean ≈ 94–98 pts (vs ~83–88 en 60 min); la mediana sube a ~80–82 pts y el p90 a ~172–181 pts → movimientos más amplios al extender el horizonte.
- Umbral no discriminante: pct_abs90_ge_thr_trade = 1.0 en todos los casos → el umbral actual queda saturado también en 90 min.
- Implicación clave: 90 min incrementa payoff potencial sin cambiar el timing óptimo; para selección efectiva conviene elevar el umbral, usar percentiles dinámicos o condicionar por volatilidad/hora.


#### 6.6.3. Top 20 por persistencia (90/60)

In [ ]:
topPersist = metrics_minute.sort_values("persist_ratio_mean", ascending=False).head(20)
display(topPersist[["hour","minute","n_persist","persist_ratio_mean","persist_ratio_p50","persist_ratio_p90"]])


,hour,minute,n_persist,persist_ratio_mean,persist_ratio_p50,persist_ratio_p90
minute_of_day,,,,,,
511,8,31,506,1.916107,1.569816,3.412186
510,8,30,438,1.833352,1.513125,3.522359
505,8,25,459,1.816268,1.472028,3.310634
509,8,29,459,1.812029,1.476923,3.419464
506,8,26,446,1.805753,1.497279,3.418934
507,8,27,436,1.797985,1.493429,3.366452
508,8,28,444,1.791898,1.480958,3.256380
512,8,32,537,1.788341,1.493274,3.089524
513,8,33,542,1.761645,1.533262,3.165263


**Comentarios:**

- Desplazamiento temporal claro: la persistencia máxima se concentra entre 08:16 y 08:39, antes del bloque de mayor magnitud/frecuencia (09:xx).

- Persistencia > 1 de forma sistemática: persist_ratio_mean ≈ 1.63–1.92, lo que indica continuidad direccional entre 60 y 90 min (el movimiento a 90 tiende a extender al de 60).

- Colas fuertes: persist_ratio_p90 ≈ ~3.0–3.5, sugiriendo episodios de fuerte extensión donde el movimiento a 90 min multiplica al de 60.

- Soporte estadístico suficiente: n_persist ≈ 430–600, no son resultados marginales.

- Lectura operativa: este bloque es óptimo para detección temprana (señales con alta probabilidad de continuación), mientras que el bloque 09:xx es óptimo para explotación de amplitud.

Implicación directa: estrategia en dos fases:

  - 08:xx → señalización/persistencia,
  - 09:xx → monetización/magnitud.

### **6.7 Interpretación mínima de cada métrica (para el informe)**


- `n_trade_60`, `n_trade_90`: cuántas veces el modelo “decidió operar” en ese minuto del día a lo largo del histórico.
- `abs*_trade_p50`: movimiento “típico” (mediana) en puntos.
- `abs*_trade_p90`: magnitud en escenarios fuertes (top 10%).
- `pct_abs*_ge_thr_trade`: proporción de señales “económicamente relevantes” (ej. ≥ 25 pts).
- `persist_ratio_*`: si es > 1, suele indicar que el movimiento a 90 min tiende a ser mayor que a 60 min (extensión).

# **7. Construcción y justificación de la ventana operativa**

**Interpretación del ranking horario y definición de ventanas operativas**


A partir del ranking por minuto del día construido en el punto anterior —basado en métricas de frecuencia, magnitud, calidad económica y persistencia— se observa que el comportamiento del mercado intradía no es homogéneo, sino que presenta regímenes temporales bien definidos.

En particular, el análisis revela dos bloques horarios diferenciados:

### **7.1 Ventana de gestación (predicción)**

El ranking de persistencia (`persist_ratio_mean`) muestra valores máximos de forma consistente en la franja 08:00–09:00, indicando que, en estos minutos, los movimientos observados a 60 minutos tienden a extenderse y amplificarse en horizontes posteriores (90 minutos).

Esta evidencia sugiere que:

- En esta franja el mercado define dirección, aunque aún no alcanza su máxima magnitud.
- La información contenida en los datos de este período es anticipatoria, ya que precede sistemáticamente a los movimientos más relevantes del día.

Por este motivo, esta franja se define como ventana de predicción, siendo adecuada para:

- Construir las variables de entrada (features) del modelo.
- Capturar señales tempranas del impulso intradía.

### **7.2 Ventana de expansión (ejecución)**

Por otro lado, el ranking de frecuencia operativa (`n_trade_60`, `n_trade_90`) y magnitud del movimiento (`abs*_trade_p50`, `abs*_trade_p90`) muestra máximos claros y recurrentes en la franja 09:00–10:00, con:

- Alta cantidad de señales por minuto.
- Movimientos típicos y extremos significativamente superiores al umbral económico definido.
- Proporción cercana al 100 % de operaciones con `|delta_pts| ≥ 25`.

Esto indica que en este bloque horario:
- El movimiento ya se encuentra activo y extendido.
- El edge es económicamente explotable, no solo estadísticamente significativo.

En consecuencia, esta franja se define como ventana de ejecución, es decir, el período en el cual una señal previamente anticipada puede materializarse en una operación.

### **7.3. Justificación del desfase temporal entre predicción y ejecución**


Un resultado central de este análisis es la existencia de un desfase temporal consistente entre:
- La franja donde se maximiza la persistencia (gestación del movimiento).
- La franja donde se maximiza la frecuencia y magnitud (expansión del movimiento).

Este desfase justifica el criterio metodológico de:

- Entrenar el modelo con datos comprendidos entre 08:00 y 09:00, y
- Utilizar sus predicciones para operar en el período 09:00–10:00.

Dicho enfoque evita el uso de información contemporánea o futura (data leakage) y alinea el diseño del modelo con la estructura temporal observada empíricamente en los datos.

### **7.4. Conclusión del punto 7**

El análisis intradía de `mnq_intraday_labeled` permite concluir que la definición de ventanas operativas no es arbitraria, sino que se encuentra respaldada por métricas objetivas extraídas del propio dataset. La separación entre una ventana de predicción y una ventana de ejecución constituye, por lo tanto, una decisión metodológica coherente con la dinámica observada del mercado y sienta las bases para el diseño experimental de las etapas posteriores del pipeline

### **7.5 Uso del `dataset mnq_intraday_labeled` bajo el criterio de ventanas operativas**

Bajo el criterio de ventanas operativas definido en los apartados anteriores, el dataset `mnq_intraday_labeled` se mantiene como la fuente completa de información intradía, sin modificaciones en su estructura original.

A partir de este dataset, se construye un conjunto de datos derivado para entrenamiento y evaluación del modelo, aplicando el siguiente esquema día a día:

- Variables de entrada (features)

  Se extraen exclusivamente de las observaciones comprendidas entre 08:00 y 09:00, correspondientes a la ventana de gestación del movimiento.
  Estas variables incluyen precios, volumen y atributos técnicos derivados, y representan la información disponible antes de la materialización del movimiento principal.

- Variables objetivo (targets)

  Se obtienen a partir de las columnas ya calculadas en mnq_intraday_labeled (por ejemplo, `delta_pts_60`, `delta_pts_90` o sus versiones discretizadas), asociadas al intervalo 09:00–10:00 del mismo día, correspondiente a la ventana de expansión.

Este diseño garantiza una separación temporal estricta entre información de entrada y variables objetivo, evitando el uso de información futura (data leakage) y alineando el entrenamiento del modelo con la estructura temporal observada empíricamente en los datos.

# **8. Bloque de código único, modular y reutilizable**

In [34]:
# ============================================================
# stage_03_target_definition (Notebook + ejecutable .py)
# Targets: trade_h (dir + base) y hold_h (extensión condicional)
# Horizontes: (60, 90)
# ============================================================

from __future__ import annotations

import os
import json
from dataclasses import dataclass
from typing import Iterable, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd


# ----------------------------
# Config
# ----------------------------
@dataclass
class Stage03Config:
    # IO
    input_path: str = "data/processed/mnq_intraday.parquet"
    output_labeled_path: str = "data/processed/mnq_intraday_labeled.parquet"
    summary_path: str = "reports/target_definition_summary.json"

    # Columns
    datetime_col: Optional[str] = None  # si None, se usa el índice si es DatetimeIndex
    date_col: str = "date"
    close_col: str = "close"

    # Targets
    horizons: Tuple[int, ...] = (60, 90)

    # Umbrales económicos (en puntos)
    base_pts: float = 25.0     # Δ_base
    ext_pts: float = 62.5      # Δ_ext

    # Output behavior
    drop_na_targets: bool = False
    timezone_expected: Optional[str] = None


# ----------------------------
# Helpers
# ----------------------------
def _ensure_parent_dir(path: str) -> None:
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)

def _as_dtindex(df: pd.DataFrame, datetime_col: Optional[str]) -> pd.DataFrame:
    dfx = df.copy()
    if datetime_col is None:
        if not isinstance(dfx.index, pd.DatetimeIndex):
            raise TypeError("Se esperaba df.index como DatetimeIndex. O indique datetime_col en config.")
        return dfx.sort_index()
    if datetime_col not in dfx.columns:
        raise KeyError(f"datetime_col='{datetime_col}' no existe en el DataFrame.")
    dfx[datetime_col] = pd.to_datetime(dfx[datetime_col], utc=False, errors="raise")
    dfx = dfx.set_index(datetime_col).sort_index()
    if not isinstance(dfx.index, pd.DatetimeIndex):
        raise TypeError("No se pudo convertir datetime_col a DatetimeIndex.")
    return dfx

def _infer_or_build_date_col(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    dfx = df.copy()
    if date_col not in dfx.columns:
        dfx[date_col] = pd.to_datetime(dfx.index.date)
    else:
        dfx[date_col] = pd.to_datetime(dfx[date_col])
    return dfx

def _basic_target_stats(x: pd.Series) -> Dict[str, Any]:
    x = x.dropna()
    if x.empty:
        return {
            "count": 0, "mean": None, "std": None, "min": None,
            "p01": None, "p05": None, "p50": None, "p95": None, "p99": None, "max": None,
        }
    return {
        "count": int(x.shape[0]),
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)) if x.shape[0] > 1 else 0.0,
        "min": float(x.min()),
        "p01": float(x.quantile(0.01)),
        "p05": float(x.quantile(0.05)),
        "p50": float(x.quantile(0.50)),
        "p95": float(x.quantile(0.95)),
        "p99": float(x.quantile(0.99)),
        "max": float(x.max()),
    }

def _pct_nans(x: pd.Series) -> float:
    return float(x.isna().mean())

def _safe_tz_name(df: pd.DataFrame) -> Optional[str]:
    try:
        return str(df.index.tz) if df.index.tz is not None else None
    except Exception:
        return None


# ----------------------------
# Core: Target Definition
# ----------------------------
def make_mnq_intraday_labeled(
    df: pd.DataFrame,
    date_col: str = "date",
    close_col: str = "close",
    horizons: Iterable[int] = (60, 90),
    base_pts: float = 25.0,
    ext_pts: float = 62.5,
    drop_na_targets: bool = False,
) -> pd.DataFrame:
    """
    Genera targets por día sin cruzar sesiones.

    Δpts_{t,h} = close_{t+h} - close_t    (shift(-h) dentro del día)

    trade_h:
      +1 si Δpts >=  base_pts
      -1 si Δpts <= -base_pts
       0 si |Δpts| < base_pts

    hold_h (condicionada a trade):
      1 si trade_h = +1 y Δpts >=  ext_pts
      1 si trade_h = -1 y Δpts <= -ext_pts
      0 en caso contrario

    Restricción: trade_h = 0 => hold_h = 0 (se cumple por construcción).
    """
    dfx = df.copy()

    if close_col not in dfx.columns:
        raise KeyError(f"Falta close_col='{close_col}'.")
    if date_col not in dfx.columns:
        raise KeyError(f"Falta date_col='{date_col}' (para no cruzar sesiones).")

    dfx = dfx.sort_index()

    for h in horizons:
        fut_close = dfx.groupby(date_col, sort=False)[close_col].shift(-h)
        delta = fut_close - dfx[close_col]
        dfx[f"delta_pts_{h}"] = delta

        # trade: dirección + umbral base
        trade = np.where(delta >= base_pts, 1, np.where(delta <= -base_pts, -1, 0)).astype("int8")
        dfx[f"trade_{h}"] = trade

        # hold: extensión condicional (solo si hay trade y en el mismo sentido)
        hold = np.zeros(len(dfx), dtype="int8")
        hold[(trade == 1) & (delta >= ext_pts)] = 1
        hold[(trade == -1) & (delta <= -ext_pts)] = 1
        dfx[f"hold_{h}"] = hold

    if drop_na_targets:
        dfx = dfx.dropna(subset=[f"delta_pts_{h}" for h in horizons])

    return dfx


# ----------------------------
# Summary (JSON report)
# ----------------------------
def build_target_definition_summary(df_labeled: pd.DataFrame, cfg: Stage03Config) -> Dict[str, Any]:
    horizons = cfg.horizons

    tzname = _safe_tz_name(df_labeled)
    dt_min = df_labeled.index.min()
    dt_max = df_labeled.index.max()
    n_rows, n_cols = map(int, df_labeled.shape)
    n_days = int(df_labeled[cfg.date_col].nunique()) if cfg.date_col in df_labeled.columns else None

    minutes_per_day = (
        df_labeled.groupby(cfg.date_col).size().describe().to_dict()
        if cfg.date_col in df_labeled.columns else None
    )

    targets: Dict[str, Any] = {}
    for h in horizons:
        key = f"h{h}"
        delta_col = f"delta_pts_{h}"
        trade_col = f"trade_{h}"
        hold_col = f"hold_{h}"

        pct_nan_delta = _pct_nans(df_labeled[delta_col])
        delta_stats = _basic_target_stats(df_labeled[delta_col])

        trade_counts = df_labeled[trade_col].value_counts(dropna=False).to_dict()
        hold_rate = float((df_labeled[hold_col] == 1).mean())

        trade_mask = df_labeled[trade_col] != 0
        hold_rate_given_trade = float(df_labeled.loc[trade_mask, hold_col].mean()) if trade_mask.any() else None

        # stats condicionadas a trade (solo donde hay trade)
        delta_on_trade = df_labeled.loc[trade_mask, delta_col]
        abs_delta_on_trade = delta_on_trade.abs()

        targets[key] = {
            "horizon_min": int(h),
            "definition": {
                "delta_pts": "close_{t+h} - close_t",
                "trade": f"+1 if delta>= {cfg.base_pts}, -1 if delta<= -{cfg.base_pts}, else 0",
                "hold": (
                    f"1 if (trade=+1 and delta>= {cfg.ext_pts}) or (trade=-1 and delta<= -{cfg.ext_pts}); else 0. "
                    "Also trade=0 => hold=0."
                ),
            },
            "columns": {"delta_pts": delta_col, "trade": trade_col, "hold": hold_col},
            "pct_nans_delta_pts": pct_nan_delta,
            "delta_pts_stats": delta_stats,
            "trade_class_counts": {str(k): int(v) for k, v in trade_counts.items()},
            "hold_rate": hold_rate,
            "hold_rate_given_trade": hold_rate_given_trade,
            "delta_pts_on_trade_stats": _basic_target_stats(delta_on_trade),
            "abs_delta_pts_on_trade_stats": _basic_target_stats(abs_delta_on_trade),
        }

    summary = {
        "stage": "stage_03_target_definition",
        "description": (
            "Definición explícita de variables objetivo (trade/hold) para horizontes 60/90. "
            "Cálculo por día sin cruzar sesiones: groupby(date) + shift(-h)."
        ),
        "input": cfg.input_path,
        "output_labeled": cfg.output_labeled_path,
        "artifacts": [cfg.summary_path],
        "params": {
            "horizons": list(cfg.horizons),
            "base_pts": cfg.base_pts,
            "ext_pts": cfg.ext_pts,
            "drop_na_targets": cfg.drop_na_targets,
            "date_col": cfg.date_col,
            "close_col": cfg.close_col,
        },
        "dataset_info": {
            "n_rows": n_rows,
            "n_cols": n_cols,
            "datetime_min": str(dt_min),
            "datetime_max": str(dt_max),
            "timezone": tzname,
            "n_days": n_days,
            "minutes_per_day_describe": minutes_per_day,
            "final_target_columns": [
                *[f"delta_pts_{h}" for h in horizons],
                *[f"trade_{h}" for h in horizons],
                *[f"hold_{h}" for h in horizons],
            ],
        },
        "targets": targets,
        "minimum_metrics": {
            "pct_nans_per_target": {k: v["pct_nans_delta_pts"] for k, v in targets.items()},
            "basic_stats_per_target": {k: v["delta_pts_stats"] for k, v in targets.items()},
        },
    }
    return summary


# ----------------------------
# Runner (Notebook y .py)
# ----------------------------
def run_stage_03(cfg: Stage03Config) -> tuple[pd.DataFrame, Dict[str, Any]]:
    if not os.path.exists(cfg.input_path):
        raise FileNotFoundError(f"Input no encontrado: {cfg.input_path}")

    df_raw = pd.read_parquet(cfg.input_path)
    df_raw = _as_dtindex(df_raw, cfg.datetime_col)

    if cfg.timezone_expected is not None:
        tzname = _safe_tz_name(df_raw)
        if tzname != cfg.timezone_expected:
            print(f"[WARN] timezone={tzname} (esperado={cfg.timezone_expected}).")

    df_raw = _infer_or_build_date_col(df_raw, cfg.date_col)

    df_labeled = make_mnq_intraday_labeled(
        df_raw,
        date_col=cfg.date_col,
        close_col=cfg.close_col,
        horizons=cfg.horizons,
        base_pts=cfg.base_pts,
        ext_pts=cfg.ext_pts,
        drop_na_targets=cfg.drop_na_targets,
    )

    _ensure_parent_dir(cfg.output_labeled_path)
    df_labeled.to_parquet(cfg.output_labeled_path, index=True)

    summary = build_target_definition_summary(df_labeled, cfg)
    _ensure_parent_dir(cfg.summary_path)
    with open(cfg.summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    return df_labeled, summary


# ============================================================
# NOTEBOOK: EJECUCIÓN
# ============================================================
cfg = Stage03Config(
    input_path=drive_path + "/data/processed/mnq_intraday.parquet",
    output_labeled_path=drive_path + "/data/processed/mnq_intraday_labeled.parquet",
    summary_path=drive_path + "/reports/target_definition_summary.json",
    horizons=(60, 90),
    base_pts=25.0,
    ext_pts=62.5,
    drop_na_targets=False,
)

df_labeled, summary = run_stage_03(cfg)

print("✅ Labeled parquet:", cfg.output_labeled_path)
print("✅ Summary JSON:", cfg.summary_path)

display(df_labeled.head(5))
display(df_labeled[[f"delta_pts_{h}" for h in cfg.horizons] +
                  [f"trade_{h}" for h in cfg.horizons] +
                  [f"hold_{h}" for h in cfg.horizons]].head(10))

min_m = summary["minimum_metrics"]
print("\n%NaNs por target (delta):")
display(pd.DataFrame.from_dict(min_m["pct_nans_per_target"], orient="index", columns=["pct_nan_delta"]))

print("\nStats básicas por target (delta):")
display(pd.DataFrame.from_dict(min_m["basic_stats_per_target"], orient="index"))



✅ Labeled parquet: /content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
✅ Summary JSON: /content/drive/MyDrive/neural_profit/reports/target_definition_summary.json


,open,high,low,close,volume,date,delta_pts_60,trade_60,hold_60,delta_pts_90,trade_90,hold_90
datetime,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,8727.75,8728.00,8727.75,8727.75,7,2019-12-23,9.00,0,0,6.00,0,0
2019-12-23 06:31:00-05:00,8727.50,8727.75,8726.50,8726.50,89,2019-12-23,9.25,0,0,7.50,0,0
2019-12-23 06:32:00-05:00,8726.50,8726.50,8725.25,8725.25,34,2019-12-23,9.75,0,0,8.00,0,0
2019-12-23 06:33:00-05:00,8725.50,8726.25,8724.75,8726.00,53,2019-12-23,8.50,0,0,8.00,0,0
2019-12-23 06:34:00-05:00,8726.00,8726.00,8726.00,8726.00,3,2019-12-23,8.00,0,0,7.75,0,0


,delta_pts_60,delta_pts_90,trade_60,trade_90,hold_60,hold_90
datetime,,,,,,
2019-12-23 06:30:00-05:00,9.00,6.00,0,0,0,0
2019-12-23 06:31:00-05:00,9.25,7.50,0,0,0,0
2019-12-23 06:32:00-05:00,9.75,8.00,0,0,0,0
2019-12-23 06:33:00-05:00,8.50,8.00,0,0,0,0
2019-12-23 06:34:00-05:00,8.00,7.75,0,0,0,0
2019-12-23 06:35:00-05:00,8.50,7.50,0,0,0,0
2019-12-23 06:36:00-05:00,8.75,8.25,0,0,0,0
2019-12-23 06:37:00-05:00,8.75,8.25,0,0,0,0
2019-12-23 06:38:00-05:00,7.50,7.00,0,0,0,0



%NaNs por target (delta):


,pct_nan_delta
h60,0.105079
h90,0.157618



Stats básicas por target (delta):


,count,mean,std,min,p01,p05,p50,p95,p99,max
h60,665833,0.481132,59.742892,-1007.25,-174.75,-95.00,2.75,86.75,155.75,1456.5
h90,626743,0.826449,74.157814,-855.50,-214.75,-118.75,3.50,108.75,190.75,1522.5
